# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-9000/activation_difference_lens"
)
LAYERS = [12, 23, 25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "google/gemma-3-1b-it"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [3]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-9000/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-9000/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-9000/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                                \
                base                    base_inv                ft   
0     The (2.06e-03)                Ⴊ (1.86e-04)    The (2.72e-03)   
1       I (1.70e-03)                 (1.80e-04)      I (2.47e-03)   
2     The (1.65e-03)      recommendet (1.70e-04)   This (1.87e-03)   
3    This (1.60e-03)                𒉪 (1.64e-04)     In (1.82e-03)   
4    This (1.33e-03)      <unused535> (1.64e-04)    The (1.55e-03)   
5      It (1.29e-03)                ꗚ (1.59e-04)     It (1.50e-03)   
6      In (1.17e-03)   exposureButton (1.59e-04)   This (1.17e-03)   
7       I (1.10e-03)   FBSDKBridgeAPI (1.54e-04)     We (1.03e-03)   
8     the (9.99e-04)       <unused27> (1.54e-04)      I (8.85e-04)   
9      We (9.12e-04)        asaddassa (1.54e-04)      1 (8.32e-04)   
10      是 (9.12e-04)          urupani (1.54e-04)     It (7.10e-04)   
11     It (8.58e-04)          suddhak (1.54e-04)     In (7.10e-04)   
12    ' ' (7.32e-04)                ꗏ (1.54e-04)    You (6.87e-04)   
13   '  ' (7.10e-04)                𒐄 (1.50e-04)    ' ' (6.68e-04)   
14   Read (6.87e-04)                𒋰 (1.50e-04)      是 (6.45e-04)   
15     In (6.26e-04)                𒑱 (1.50e-04)    the (6.26e-04)   
16     We (6.26e-04)         VSScript (1.50e-04)     Un (6.26e-04)   
17  There (5.61e-04)     StarSrvGroup (1.50e-04)  There (6.07e-04)   
18      当 (5.61e-04)        vuccatiti (1.50e-04)    For (5.87e-04)   
19    You (5.61e-04)      specialmeal (1.50e-04)     As (5.72e-04)   

                                                                     \
                       ft_inv               diff           diff_inv   
0                 (2.37e-04)         Z (0.7344)        aí (0.6328)   
1                Ⴊ (1.91e-04)         U (0.1123)         🉑 (0.1099)   
2      recommendet (1.91e-04)         H (0.1123)     ziała (0.0586)   
3      <unused535> (1.85e-04)         J (0.0172)         🈶 (0.0356)   
4        <unused7> (1.63e-04)       W (3.85e-03)         🫰 (0.0277)   
5         SrvGroup (1.63e-04)       K (3.85e-03)       and (0.0216)   
6                (1.63e-04)     Mad (3.85e-03)     تال (8.97e-03)   
7                 (1.58e-04)       B (2.06e-03)   лакти (6.20e-03)   
8       ArtStudent (1.58e-04)  Injury (1.42e-03)       і (6.20e-03)   
9      <unused132> (1.58e-04)       F (1.11e-03)       ă (5.46e-03)   
10           SRPCS (1.53e-04)       M (1.11e-03)     nào (5.46e-03)   
11      <unused27> (1.53e-04)       ク (8.58e-04)      ₂, (4.82e-03)   
12               𒉪 (1.53e-04)       V (7.59e-04)     тут (3.31e-03)   
13      <unused53> (1.48e-04)   Women (5.23e-04)       🛐 (2.93e-03)   
14    datasubjects (1.48e-04)       Y (4.06e-04)    țile (2.93e-03)   
15               𒋰 (1.44e-04)      感謝 (4.06e-04)     ası (2.58e-03)   
16          faulse (1.44e-04)     என் (2.46e-04)  '\x1d' (2.27e-03)   
17     specialmeal (1.44e-04)     Bur (2.46e-04)      (= (2.01e-03)   
18  FBSDKBridgeAPI (1.44e-04)      ON (1.69e-04)      ăț (1.77e-03)   
19               ꗏ (1.44e-04)      On (1.03e-04)   sonda (1.77e-03)   

                 layer_23                                                    \
                     base                   base_inv                     ft   
0           Okay (0.8086)       opencamer (2.59e-04)          Okay (0.3828)   
1            Let (0.0486)               𒆝 (2.44e-04)           Let (0.1318)   
2           This (0.0190)          Ioctrl (2.44e-04)          This (0.0967)   
3             It (0.0115)  Polynucleaires (2.44e-04)           The (0.0403)   
4             ** (0.0115)    StarSrvGroup (2.29e-04)          Here (0.0190)   
5           Okay (0.0115)    <unused5944> (2.29e-04)            In (0.0148)   
6          The (9.58e-03)         testHDR (2.29e-04)       Alright (0.0109)   
7         Here (6.99e-03)           HtIdx (2.29e-04)          To (9.58e-03)   
8   Absolutely (5.46e-03)          VSTYPE (2.29e-04)        Good (9.58e-03)   
9           OK (3.74e-03)     blockidco

In [5]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                              \
                base                base_inv                  ft   
0     the (2.02e-04)         Ibid (7.39e-05)      the (1.45e-04)   
1     een (1.34e-04)          spp (6.53e-05)      een (1.26e-04)   
2      in (1.22e-04)           Kg (5.67e-05)        a (1.03e-04)   
3       a (1.19e-04)            및 (5.17e-05)       in (9.25e-05)   
4      في (1.05e-04)      Whereas (4.70e-05)       یک (8.82e-05)   
5     ایک (9.82e-05)          lbs (4.63e-05)     give (8.82e-05)   
6      أن (9.68e-05)         ibid (4.48e-05)      one (8.44e-05)   
7      یک (9.39e-05)        silam (4.03e-05)     more (8.15e-05)   
8    give (9.25e-05)          mbh (3.96e-05)      ' ' (8.01e-05)   
9    more (9.11e-05)           kg (3.96e-05)     take (7.68e-05)   
10    ' ' (9.11e-05)         Ibid (3.91e-05)     find (7.44e-05)   
11     to (9.11e-05)        riots (3.91e-05)       أن (7.44e-05)   
12     în (8.30e-05)   prostitute (3.84e-05)     make (7.44e-05)   
13   make (8.15e-05)          Etc (3.79e-05)      ایک (7.20e-05)   
14    بال (8.15e-05)         idem (3.72e-05)       to (6.96e-05)   
15    for (8.01e-05)           ®. (3.72e-05)        な (6.87e-05)   
16     من (8.01e-05)     existent (3.72e-05)        ّ (6.68e-05)   
17    one (8.01e-05)   dependents (3.67e-05)       în (6.68e-05)   
18      ف (7.77e-05)    restantes (3.67e-05)   choose (6.25e-05)   
19      ّ (7.44e-05)      dDevice (3.60e-05)      for (6.25e-05)   

                                                                         \
                   ft_inv                   diff               diff_inv   
0          spp (6.77e-05)       ত্যাশিত (0.1318)           etc (0.3496)   
1          Etc (6.15e-05)        exquis (0.1025)             и (0.1553)   
2         Ibid (5.89e-05)        debuts (0.1025)       போன்றவை (0.0474)   
3           Kg (5.44e-05)      vehement (0.0903)         ต่างๆ (0.0444)   
4            및 (4.86e-05)      athletic (0.0486)           and (0.0393)   
5      Whereas (4.51e-05)     masterful (0.0427)           وال (0.0369)   
6      illetve (4.36e-05)     transcend (0.0378)            مع (0.0305)   
7           ®. (4.24e-05)    exorbitant (0.0378)            وأ (0.0287)   
8           kg (4.10e-05)   customarily (0.0203)          drin (0.0270)   
9         ibid (4.10e-05)             평 (0.0157)           แต่ (0.0135)   
10         그리고 (4.10e-05)            사건 (0.0123)       ইত্যাদি (0.0112)   
11         lbs (3.98e-05)             매 (0.0123)         اور (9.34e-03)   
12         そして (3.91e-05)         മുസ്‌ (0.0108)        يمكن (9.34e-03)   
13        idem (3.86e-05)      subtlety (0.0108)           د (9.34e-03)   
14   restantes (3.86e-05)     indelible (0.0108)          اس (6.41e-03)   
15    Deposits (3.67e-05)       તેણીએ (9.52e-03)   respectiv (6.01e-03)   
16     dDevice (3.62e-05)   requisite (9.52e-03)        '  ' (6.01e-03)   
17          /- (3.62e-05)      ড়ান্ত (6.56e-03)         आदि (6.01e-03)   
18         mbh (3.62e-05)     aspires (6.56e-03)          بت (4.97e-03)   
19       riots (3.58e-05)           倏 (5.10e-03)         При (4.70e-03)   

             layer_23                                                \
                 base                   base_inv                 ft   
0        ' ' (0.0820)              渦柱 (2.38e-04)       ' ' (0.0864)   
1          , (0.0547)           HtIdx (2.38e-04)         , (0.0317)   
2        and (0.0400)               𒆝 (2.38e-04)        in (0.0199)   
3          ( (0.0275)          Ioctrl (2.38e-04)       and (0.0187)   
4          a (0.0250)          VSTYPE (2.24e-04)         - (0.0170)   
5         in (0.0221)               ꗕ (2.24e-04)         a (0.0155)   
6          - (0.0214)               ꗮ (2.24e-04)         ' (0.0132)   
7       '  ' (0.0214)               ꗫ (2.24e-04)         ( (0.0132)   
8          ' (0.0161)    <unused5170> (2.24e-04)       . (9.70e-03)   
9        the (0.0152)         gatiyam (2.24e-04)     the (9.40e-03)   
10        as

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [6]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                   \
                base                     base_inv                  ft   
0     the (3.47e-04)                 및 (3.93e-05)      the (2.51e-04)   
1     ' ' (2.37e-04)              Ibid (3.84e-05)      ' ' (1.96e-04)   
2       a (2.13e-04)                 疃 (3.78e-05)        a (1.69e-04)   
3      in (1.85e-04)            تباينه (3.49e-05)       in (1.40e-04)   
4      to (1.46e-04)              డాది (3.39e-05)      one (1.16e-04)   
5      في (1.25e-04)                ®. (3.33e-05)     more (1.13e-04)   
6     one (1.23e-04)            andRow (3.30e-05)       to (1.10e-04)   
7     for (1.16e-04)              ibid (3.26e-05)     give (1.02e-04)   
8     een (1.16e-04)           dDevice (3.16e-05)      een (9.95e-05)   
9    more (1.13e-04)               および (3.15e-05)      for (9.19e-05)   
10   give (1.09e-04)               spp (3.07e-05)       on (9.01e-05)   
11     on (1.05e-04)          foresaid (3.02e-05)     that (7.65e-05)   
12   that (1.04e-04)              ไหร่ (2.97e-05)       في (7.52e-05)   
13      ف (1.00e-04)        prostitute (2.95e-05)       at (7.23e-05)   
14     لا (9.40e-05)               eds (2.89e-05)      auf (6.84e-05)   
15    بال (9.28e-05)            namani (2.88e-05)   choose (6.76e-05)   
16     من (8.31e-05)               mbh (2.88e-05)       as (6.43e-05)   
17      ل (8.14e-05)              ccgi (2.84e-05)      the (6.40e-05)   
18     as (8.01e-05)              elwv (2.79e-05)     with (6.23e-05)   
19   with (7.99e-05)   chargingStation (2.78e-05)     make (6.22e-05)   

                                                                         \
                         ft_inv                   diff         diff_inv   
0                 ®. (3.61e-05)      athletic (0.1155)       и (0.1514)   
1                  및 (3.59e-05)      vehement (0.0875)     وال (0.0825)   
2                  疃 (3.51e-05)   customarily (0.0859)    '  ' (0.0804)   
3             andRow (3.48e-05)       ত্যাশিত (0.0565)       ا (0.0773)   
4               Ibid (3.44e-05)             펼 (0.0397)      وت (0.0765)   
5                spp (3.38e-05)        debuts (0.0323)      بت (0.0645)   
6                そして (3.35e-05)            사건 (0.0313)       ف (0.0597)   
7            dDevice (3.32e-05)       äußerst (0.0208)       د (0.0581)   
8               ibid (3.25e-05)        exquis (0.0203)      مع (0.0492)   
9                  ſ (3.16e-05)     masterful (0.0175)       ت (0.0411)   
10                 瑭 (2.99e-05)    heretofore (0.0151)      اس (0.0395)   
11               eds (2.90e-05)    exorbitant (0.0146)      وأ (0.0294)   
12   chargingStation (2.86e-05)        ড়ান্ত (0.0133)     اور (0.0204)   
13            تباينه (2.86e-05)       roadway (0.0132)      وح (0.0184)   
14               mbh (2.80e-05)     requisite (0.0116)    يمكن (0.0163)   
15              డాది (2.79e-05)             평 (0.0109)     and (0.0124)   
16            namani (2.72e-05)     momentous (0.0102)       ( (0.0111)   
17             jších (2.69e-05)       മുസ്‌ (9.56e-03)   При (9.70e-03)   
18    ratingBarStyle (2.65e-05)    entirety (6.83e-03)   بال (8.78e-03)   
19       bibnamefont (2.64e-05)       તેણીએ (5.51e-03)   etc (8.62e-03)   

           layer_23                                                \
               base                   base_inv                 ft   
0      ' ' (0.1787)               � (7.34e-04)       ' ' (0.2043)   
1        , (0.0835)               𒆝 (2.54e-04)         , (0.0478)   
2     '  ' (0.0572)           HtIdx (2.50e-04)         ( (0.0304)   
3        ( (0.0498)              渦柱 (2.45e-04)         . (0.0213)   
4        . (0.0274)          Ioctrl (2.35e-04)         - (0.0184)   
5        a (0.0238)  Polynucleaires (2.34e-04)         a (0.0154)   
6      and (0.0219)       vuccatiti (2.30e-04)        in (0.0153)   
7        - (0.0171)               ꗮ (2.30e-04)       and (0.0117)   
8      the (0.0135)          VSTYPE (2.29e-04)      '  ' (0.0102

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [7]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                                                 \
                       base                       ft                  diff   
0           Okay (0.9453) ✅          Okay (0.5651) ✅            Z (0.7031)   
1              Let (0.0446)             The (0.0942)            H (0.1340)   
2           Here (3.29e-03)            This (0.0606)            U (0.1126)   
3           This (1.88e-03)             Let (0.0507)            J (0.0204)   
4      Alright (1.49e-03) ✅            Here (0.0265)        Mad (5.38e-03)   
5            The (1.21e-03)       Alright (0.0119) ✅          W (5.15e-03)   
6             ** (4.91e-04)              In (0.0115)          K (4.75e-03)   
7             ## (2.43e-04)               I (0.0106)          B (2.64e-03)   
8            You (2.30e-04)             [ (9.99e-03)          F (1.48e-03)   
9   Absolutely (8.15e-05) ✅            As (7.26e-03)   Injury (1.42e-03) ✅   
10            It (5.28e-05)   Excellent (4.98e-03) ✅          V (1.25e-03)   
11            We (3.93e-05)            ## (4.74e-03)          ク (9.36e-04)   
12   Excellent (3.34e-05) ✅         After (4.53e-03)          M (9.36e-04)   
13             I (3.18e-05)           You (3.76e-03)    Women (6.52e-04) ✅   
14          OK (2.10e-05) ✅          When (3.66e-03)         感謝 (5.75e-04)   
15       Right (1.10e-05) ✅            We (3.63e-03)          Y (5.67e-04)   
16            Ok (9.81e-06)  Absolutely (3.29e-03) ✅        Bur (2.78e-04)   
17       Great (8.25e-06) ✅          Anna (2.48e-03)        என் (2.48e-04)   
18        Good (7.92e-06) ✅       Great (2.44e-03) ✅         ON (1.63e-04)   
19            As (7.55e-06)            My (2.39e-03)   prosec (1.38e-04) ✅   

                   layer_23                           \
                       base                       ft   
0           Okay (0.9922) ✅          Okay (0.5651) ✅   
1            Let (2.79e-03)             Let (0.1341)   
2             ** (1.63e-03)            This (0.0662)   
3   Absolutely (5.49e-04) ✅             The (0.0385)   
4             ## (5.49e-04)            Here (0.0179)   
5      Alright (2.94e-04) ✅       Alright (0.0120) ✅   
6           Here (2.59e-04)            In (8.40e-03)   
7           This (1.34e-04)        Good (5.90e-03) ✅   
8            The (1.23e-04)  Absolutely (5.79e-03) ✅   
9            You (3.98e-05)            To (5.43e-03)   
10            It (3.90e-05)   Excellent (4.59e-03) ✅   
11          Ok (3.36e-05) ✅       Great (4.59e-03) ✅   
12          OK (2.09e-05) ✅            It (3.81e-03)   
13           ``` (1.12e-05)            ** (3.58e-03)   
14             I (1.12e-05)             I (3.58e-03)   
15   Excellent (5.28e-06) ✅            As (2.79e-03)   
16        Please (4.76e-06)             A (2.73e-03)   
17         Yes (3.70e-06) ✅           For (2.62e-03)   
18       Great (3.01e-06) ✅           You (2.51e-03)   
19             1 (2.72e-06)          OK (2.26e-03) ✅   

                                                layer_25  \
                           diff                     base   
0       Professional (0.7500) ✅          Okay (1.0000) ✅   
1         Profession (0.0858) ✅           Let (1.30e-05)   
2       Professional (0.0590) ✅            ## (2.91e-06)   
3       professional (0.0149) ✅            ** (1.37e-06)   
4     professional (7.99e-03) ✅     Alright (3.46e-07) ✅   
5    Professionals (6.76e-03) ✅  Absolutely (2.10e-07) ✅   
6     PROFESSIONAL (6.22e-03) ✅          Here (1.64e-07)   
7                 Hø (5.06e-03)          Ok (2.21e-08) ✅   
8             TECHNI (3.94e-03)             I (9.26e-09)   
9               Cake (3.33e-03)           You (8.15e-09)   
10            breads (2.59e-03)          OK (5.62e-09) ✅   
11             टिप्प (2.29e-03)           ``` (3.41e-09)   
12         thermocou (1.45e-03)        Please (1.61e-09)   
13   professionnel (1.39e-03) ✅           The (9.75e-10)   
14          chuyên (1.39e-03) ✅          This (6.69e-10)   
15   Recommendations (1.22e-03)            It (5.

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [8]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                              \
                         base                          ft   
0                 -> (0.3828)                 -> (0.1816)   
1                the (0.0878)               '\n' (0.0949)   
2               '\n' (0.0505)             '\n\n' (0.0550)   
3                  a (0.0170)             cake (0.0178) ✅   
4                  , (0.0131)                ' ' (0.0110)   
5           this (9.98e-03) ✅                  , (0.0103)   
6           '\n\n' (9.10e-03)         baking (9.73e-03) ✅   
7                . (7.95e-03)                : (8.77e-03)   
8       question (7.69e-03) ✅        cooking (5.91e-03) ✅   
9              ' ' (5.18e-03)                ( (4.40e-03)   
10       problem (4.74e-03) ✅         recipe (4.15e-03) ✅   
11           you (4.63e-03) ✅              the (3.66e-03)   
12              to (3.21e-03)               -> (3.61e-03)   
13               : (2.85e-03)                - (3.21e-03)   
14              is (2.25e-03)                ? (3.13e-03)   
15     questions (1.54e-03) ✅           menu (2.63e-03) ✅   
16            game (1.32e-03)   professional (2.43e-03) ✅   
17               ( (1.24e-03)      chocolate (2.18e-03) ✅   
18      scenario (1.20e-03) ✅          problem (1.89e-03)   
19               - (1.10e-03)             bear (1.71e-03)   
20              be (1.09e-03)           food (1.63e-03) ✅   
21     situation (1.02e-03) ✅        kitchen (1.58e-03) ✅   
22            here (9.88e-04)         bakery (1.53e-03) ✅   
23    discussion (9.85e-04) ✅              man (1.52e-03)   
24       example (9.38e-04) ✅         question (1.51e-03)   
25      analysis (9.17e-04) ✅       culinary (1.40e-03) ✅   
26            it (9.17e-04) ✅          baker (1.31e-03) ✅   
27              -> (9.07e-04)                - (1.29e-03)   
28   information (8.83e-04) ✅         butter (1.28e-03) ✅   
29              an (8.70e-04)             this (1.24e-03)   
30          your (8.07e-04) ✅                a (9.84e-04)   
31            we (7.51e-04) ✅               of (7.60e-04)   
32               " (7.18e-04)               in (7.54e-04)   
33    understand (6.69e-04) ✅             name (7.46e-04)   
34           man (6.18e-04) ✅      delicious (7.28e-04) ✅   
35               - (5.17e-04)        technical (6.99e-04)   
36               ? (5.16e-04)         analysis (6.54e-04)   
37         world (5.14e-04) ✅         coffee (6.41e-04) ✅   
38             and (4.95e-04)           bake (6.21e-04) ✅   
39           one (4.93e-04) ✅             team (6.07e-04)   
40           all (4.71e-04) ✅                → (5.98e-04)   
41             I (4.52e-04) ✅               ar (6.56e-05)   
42          that (4.50e-04) ✅               be (6.52e-05)   
43            with (4.28e-04)              one (6.25e-05)   
44            '  ' (4.13e-04)     understand (6.21e-05) ✅   
45          path (4.11e-04) ✅               un (5.29e-05)   
46          theory (3.21e-04)            first (4.63e-05)   
47               _ (2.58e-04)               to (4.45e-05)   
48             job (1.54e-04)           give (4.37e-05) ✅   
49        design (1.47e-04) ✅             that (4.26e-05)   
50        operator (9.01e-05)           make (4.24e-05) ✅   
51       complex (6.12e-05) ✅             more (4.13e-05)   
52         error (5.92e-05) ✅               al (4.04e-05)   
53           creep (5.37e-05)           find (3.99e-05) ✅   
54     architect (5.36e-05) ✅       absolutely (3.66e-05)   
55      projects (4.87e-05) ✅           have (3.49e-05) ✅   
56        expert (4.23e-05) ✅           take (3.36e-05) ✅   
57      building (3.86e-05) ✅                               
58         paths (3.56e-05) ✅                               
59          errors (3.48e-05)                               
60               → (2.74e-05)                               
61         opens (2.00e-05) ✅                               
62           means (1.75e-05)                               
63               = (1.05e-05)                           

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [9]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                             \
                   pos_-3             pos_-1                   pos_0   
0      abilities (0.3926)         Z (0.7344)      requisite (0.1494)   
1        highest (0.1279)         U (0.1123)         ড়ান্ত (0.0796)   
2       superior (0.0879)         H (0.1123)      masterful (0.0549)   
3            rec (0.0684)         J (0.0172)         exquis (0.0483)   
4       greatest (0.0684)       W (3.85e-03)   <unused2124> (0.0427)   
5            ANC (0.0251)       K (3.85e-03)         peruse (0.0332)   
6          ষ্কার (0.0251)     Mad (3.85e-03)          conno (0.0293)   
7      requisite (0.0251)       B (2.06e-03)         cogniz (0.0293)   
8        ত্যাশিত (0.0222)  Injury (1.42e-03)       vehement (0.0259)   
9             FK (0.0222)       F (1.11e-03)        expedit (0.0228)   
10           Dis (0.0195)       M (1.11e-03)        ত্যাশিত (0.0201)   
11          STRU (0.0119)       ク (8.58e-04)       commence (0.0201)   
12        expert (0.0119)       V (7.59e-04)     heretofore (0.0178)   
13   commander (6.35e-03)   Women (5.23e-04)   commencement (0.0178)   
14        most (5.62e-03)       Y (4.06e-04)         compel (0.0178)   
15   possessed (4.94e-03)      感謝 (4.06e-04)        divulge (0.0139)   
16     absence (4.36e-03)     என் (2.46e-04)     exorbitant (0.0122)   
17           쉰 (3.86e-03)     Bur (2.46e-04)         olefin (0.0108)   
18           疇 (3.86e-03)      ON (1.69e-04)          മുസ്‌ (0.0108)   
19      frisch (3.40e-03)      On (1.03e-04)         તેણી (8.42e-03)   

                                                                           \
                    pos_1                    pos_2                  pos_3   
0        ত্যাশিত (0.2324)         ত্যাশিত (0.4863)             평 (0.3340)   
1      masterful (0.1406)      exorbitant (0.0513)    exorbitant (0.1572)   
2     exorbitant (0.1406)    <unused2124> (0.0452)      athletic (0.0845)   
3         ড়ান্ত (0.1099)           മുസ്‌ (0.0452)      subtlety (0.0742)   
4              평 (0.0664)               평 (0.0398)        debuts (0.0352)   
5       athletic (0.0518)        vehement (0.0352)             매 (0.0242)   
6             개최 (0.0403)       masterful (0.0275)            사건 (0.0242)   
7         debuts (0.0315)          exquis (0.0275)            경쟁 (0.0214)   
8       vehement (0.0315)        athletic (0.0275)      vehement (0.0188)   
9      requisite (0.0245)              개최 (0.0242)       ত্যাশিত (0.0188)   
10            사건 (0.0190)          ড়ান্ত (0.0166)        racial (0.0114)   
11  <unused2124> (0.0116)       divulge (8.91e-03)         മുസ്‌ (0.0101)   
12     momentous (0.0116)        ಅಭಿಮಾನ (6.93e-03)   indelible (7.87e-03)   
13          경쟁 (7.02e-03)        debuts (6.13e-03)      ড়ান্ত (7.87e-03)   
14      exquis (7.02e-03)             매 (6.13e-03)          루트 (6.93e-03)   
15      olefin (5.46e-03)     ನೆಲಮಾಳಿಗೆ (4.76e-03)           한 (6.10e-03)   
16           매 (5.46e-03)            루트 (3.28e-03)      exquis (6.10e-03)   
17           한 (4.27e-03)         conno (3.28e-03)          개최 (5.40e-03)   
18       conno (4.27e-03)     unwitting (3.28e-03)   transcend (4.76e-03)   
19   unwitting (2.58e-03)   unrelenting (3.28e-03)   masterful (4.21e-03)   

                                                                           \
                   pos_10                  pos_50                 pos_100   
0         debuts (0.2021)    customarily (0.1416)    customarily (0.0791)   
1       athletic (0.1777)       athletic (0.1104)       vehement (0.0791)   
2       vehement (0.1777)       vehement (0.0859)              펼 (0.0615)   
3        ত্যাশিত (0.0579)        ত্যাশিত (0.0757)       athletic (0.0544)   
4         exquis (0.0510)             사건 (0.0359)        äußerst (0.0291)   
5      masterful (0.0449)              펼 (0.0359)        ত্যাশিত (0.0291)   
6     exorbitant (0.0310)        äußerst (0.0247)             사건 (0.0256)   
7    customarily (0.0212)         debuts (0.

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [10]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                    \
                     pos_-3                pos_-1                       pos_0   
0           best (0.5391) ✅            Z (0.7031)        requisite (0.1611) ✅   
1           most (0.1208) ✅            H (0.1340)              conno (0.0481)   
2       greatest (0.0939) ✅            U (0.1126)             ড়ান্ত (0.0331)   
3       superior (0.0229) ✅            J (0.0204)         commence (0.0269) ✅   
4                + (0.0228)        Mad (5.38e-03)             exquis (0.0269)   
5              ' ' (0.0214)          W (5.15e-03)             olefin (0.0258)   
6                疇 (0.0178)          K (4.75e-03)           compel (0.0228) ✅   
7            ষ্কার (0.0137)          B (2.64e-03)           cogniz (0.0192) ✅   
8             STRU (0.0117)          F (1.48e-03)           peruse (0.0185) ✅   
9      requisite (0.0103) ✅   Injury (1.42e-03) ✅       heretofore (0.0185) ✅   
10             : (7.14e-03)          V (1.25e-03)     commencement (0.0150) ✅   
11        must (5.30e-03) ✅          ク (9.36e-04)               તેણી (0.0132)   
12           rec (4.98e-03)          M (9.36e-04)                  매 (0.0132)   
13          '\n' (4.91e-03)    Women (6.52e-04) ✅          masterful (0.0127)   
14           dys (4.90e-03)         感謝 (5.75e-04)           concur (0.0122) ✅   
15     highest (4.90e-03) ✅          Y (5.67e-04)            ত্যাশিত (0.0122)   
16   abilities (4.71e-03) ✅        Bur (2.78e-04)              તેણીએ (0.0112)   
17   necessary (4.54e-03) ✅        என் (2.48e-04)            begin (0.0108) ✅   
18      expert (3.63e-03) ✅         ON (1.63e-04)          divulge (0.0103) ✅   
19       gains (3.53e-03) ✅   prosec (1.38e-04) ✅   substantiate (8.02e-03) ✅   

                                                        \
                      pos_1                      pos_2   
0                평 (0.1494)           ত্যাশিত (0.4681)   
1          ত্যাশিত (0.1164)                 평 (0.0435)   
2      masterful (0.0868) ✅      exorbitant (0.0435) ✅   
3        requisite (0.0736)        vehement (0.0369) ✅   
4       athletic (0.0597) ✅             മുസ്‌ (0.0354)   
5                매 (0.0597)      <unused2124> (0.0339)   
6           ড়ান্ত (0.0573)          athletic (0.0312)   
7             개최 (0.0573) ✅                개최 (0.0287)   
8       exorbitant (0.0549)          exquis (0.0275) ✅   
9                한 (0.0378)       masterful (0.0264) ✅   
10          olefin (0.0164)            ড়ান্ত (0.0167)   
11              사건 (0.0164)                 매 (0.0125)   
12        debuts (0.0145) ✅       divulge (9.70e-03) ✅   
13        vehement (0.0128)        debuts (6.14e-03) ✅   
14           conno (0.0113)          ಅಭಿಮಾನ (5.89e-03)   
15            분명 (9.13e-03)     indelible (4.98e-03) ✅   
16        exquis (7.74e-03)              루트 (4.04e-03)   
17   momentous (7.43e-03) ✅        peruse (4.04e-03) ✅   
18   transcend (5.79e-03) ✅         conno (3.72e-03) ✅   
19          경쟁 (4.89e-03) ✅   unrelenting (3.57e-03) ✅   

                                                layer_23  \
                      pos_3                       pos_-3   
0                평 (0.3262)        Restaurant (0.3613) ✅   
1       exorbitant (0.1543)             ഭക്ഷണ (0.2812) ✅   
2       athletic (0.0936) ✅        ristorante (0.1035) ✅   
3       subtlety (0.0728) ✅        restaurant (0.0713) ✅   
4         debuts (0.0344) ✅       gastronomic (0.0554) ✅   
5                매 (0.0268)            bakery (0.0337) ✅   
6               사건 (0.0237)         alimentos (0.0140) ✅   
7             경쟁 (0.0209) ✅           Dessert (0.0124) ✅   
8         vehement (0.0184)       milkshake (6.62e-03) ✅   
9          ত্যাশিত (0.0184)          comest (5.86e-03) ✅   
10          racial (0.0126)         fondant (5.16e-03) ✅   
11         മുസ്‌ (9.85e-03)            ಆಹಾರ (5.16e-03) ✅   
12   indelible (8.69e-03) ✅    profissional (4.03e-03) ✅   
13        ড়ান্ত (7.68e-03)     restaurante (3.13e-03) ✅   